# 1. Import Libraries


In [14]:
import pandas as pd
import numpy as np


# 2. Load Dataset


In [15]:
file_path = "../data/raw/survey_results_public.csv"
df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)


Dataset shape: (89184, 84)


# 3. Basic Dataset Overview

In [16]:
df.head()

,ResponseId,Q120,MainBranch,Age,Employment,RemoteWork,CodingActivities,EdLevel,LearnCode,LearnCodeOnline,...,Frequency_1,Frequency_2,Frequency_3,TimeSearching,TimeAnswering,ProfessionalTech,Industry,SurveyLength,SurveyEase,ConvertedCompYearly
0,1,I agree,None of these,18-24 years old,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,I agree,I am a developer by profession,25-34 years old,"Employed, full-time",Remote,Hobby;Contribute to open-source projects;Boots...,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Books / Physical media;Colleague;Friend or fam...,Formal documentation provided by the owner of ...,...,1-2 times a week,10+ times a week,Never,15-30 minutes a day,15-30 minutes a day,DevOps function;Microservices;Automated testin...,"Information Services, IT, Software Development...",Appropriate in length,Easy,285000.0
2,3,I agree,I am a developer by profession,45-54 years old,"Employed, full-time","Hybrid (some remote, some in-person)",Hobby;Professional development or self-paced l...,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Books / Physical media;Colleague;On the job tr...,Formal documentation provided by the owner of ...,...,6-10 times a week,6-10 times a week,3-5 times a week,30-60 minutes a day,30-60 minutes a day,DevOps function;Microservices;Automated testin...,"Information Services, IT, Software Development...",Appropriate in length,Easy,250000.0
3,4,I agree,I am a developer by profession,25-34 years old,"Employed, full-time","Hybrid (some remote, some in-person)",Hobby,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Colleague;Friend or family member;Other online...,Formal documentation provided by the owner of ...,...,1-2 times a week,10+ times a week,1-2 times a week,15-30 minutes a day,30-60 minutes a day,Automated testing;Continuous integration (CI) ...,NaN,Appropriate in length,Easy,156000.0
4,5,I agree,I am a developer by profession,25-34 years old,"Employed, full-time;Independent contractor, fr...",Remote,Hobby;Contribute to open-source projects;Profe...,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Books / Physical media;Online Courses or Certi...,Formal documentation provided by the owner of ...,...,1-2 times a week,1-2 times a week,3-5 times a week,60-120 minutes a day,30-60 minutes a day,Microservices;Automated testing;Observability ...,Other,Appropriate in length,Neither easy nor difficult,23456.0


In [17]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 89184 entries, 0 to 89183
Data columns (total 84 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   ResponseId                           89184 non-null  int64  
 1   Q120                                 89184 non-null  object 
 2   MainBranch                           89184 non-null  object 
 3   Age                                  89184 non-null  object 
 4   Employment                           87898 non-null  object 
 5   RemoteWork                           73810 non-null  object 
 6   CodingActivities                     73764 non-null  object 
 7   EdLevel                              87973 non-null  object 
 8   LearnCode                            87663 non-null  object 
 9   LearnCodeOnline                      70084 non-null  object 
 10  LearnCodeCoursesCert                 37076 non-null  object 
 11  YearsCode                   

# 4. Column Selection for Modeling

In [18]:
selected_columns = [
    "ConvertedCompYearly",
    "YearsCodePro",
    "Country",
    "EdLevel",
    "Employment"
]

df_selected = df[selected_columns]

print("Selected shape:", df_selected.shape)
df_selected.head()


Selected shape: (89184, 5)


,ConvertedCompYearly,YearsCodePro,Country,EdLevel,Employment
0,NaN,NaN,NaN,NaN,NaN
1,285000.0,9,United States of America,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Employed, full-time"
2,250000.0,23,United States of America,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Employed, full-time"
3,156000.0,7,United States of America,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Employed, full-time"
4,23456.0,4,Philippines,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Employed, full-time;Independent contractor, fr..."


# 5. Missing Value Analysis (Selected Columns)


In [19]:
df_selected.isna().sum()


ConvertedCompYearly    41165
YearsCodePro           23048
Country                 1211
EdLevel                 1211
Employment              1286
dtype: int64

# 6. Remove Rows with Missing Target

In [20]:
df_selected = df_selected.dropna(subset=["ConvertedCompYearly"])

print("Shape after removing missing salary:", df_selected.shape)
df_selected.isna().sum()


Shape after removing missing salary: (48019, 5)


ConvertedCompYearly      0
YearsCodePro           194
Country                  0
EdLevel                  0
Employment              12
dtype: int64

# 7. Remove Remaining Small Missing Values


In [21]:
df_selected = df_selected.dropna()

print("Final shape after removing all missing:", df_selected.shape)
df_selected.isna().sum()


Final shape after removing all missing: (47813, 5)


ConvertedCompYearly    0
YearsCodePro           0
Country                0
EdLevel                0
Employment             0
dtype: int64

# 8. Clean YearsCodePro Column

In [22]:
def clean_experience(value):
    if value == "Less than 1 year":
        return 0.5
    elif value == "More than 50 years":
        return 50
    else:
        return float(value)

df_selected["YearsCodePro"] = df_selected["YearsCodePro"].apply(clean_experience)

df_selected["YearsCodePro"].dtype


dtype('float64')

In [23]:
df_selected["YearsCodePro"].describe()


count    47813.000000
mean        10.758915
std          8.667537
min          0.500000
25%          4.000000
50%          8.000000
75%         15.000000
max         50.000000
Name: YearsCodePro, dtype: float64

# 9. Salary Distribution Analysis

In [24]:
df_selected["ConvertedCompYearly"].describe()


count    4.781300e+04
mean     1.031599e+05
std      6.828201e+05
min      1.000000e+00
25%      4.390700e+04
50%      7.496300e+04
75%      1.217260e+05
max      7.435143e+07
Name: ConvertedCompYearly, dtype: float64

In [31]:
salary_cap = df_selected["ConvertedCompYearly"].quantile(0.99)

df_selected = df_selected[df_selected["ConvertedCompYearly"] <= salary_cap]

print("Shape after removing top 1%:", df_selected.shape)


Shape after removing top 1%: (46902, 5)


In [32]:
df_selected.isna().sum()


ConvertedCompYearly    0
YearsCodePro           0
Country                0
EdLevel                0
Employment             0
dtype: int64

In [33]:
df_selected["ConvertedCompYearly"].describe(percentiles=[0.90, 0.95, 0.99])


count     46902.000000
mean      86464.812524
std       60578.532012
min           1.000000
50%       74483.000000
90%      171343.000000
95%      200301.150000
99%      274437.920000
max      310345.000000
Name: ConvertedCompYearly, dtype: float64

# 10. Log Transform Salary

In [34]:
import numpy as np

df_selected["LogSalary"] = np.log(df_selected["ConvertedCompYearly"])

df_selected["LogSalary"].describe()


count    46902.000000
mean        10.979550
std          1.212016
min          0.000000
25%         10.672745
50%         11.218326
75%         11.686770
max         12.645440
Name: LogSalary, dtype: float64

In [35]:
df_selected = df_selected[df_selected["ConvertedCompYearly"] >= 1000]

print("Shape after removing very low salaries:", df_selected.shape)


Shape after removing very low salaries: (46224, 6)


# 11. Define Features and Target

In [36]:
X = df_selected[["YearsCodePro", "Country", "EdLevel", "Employment"]]
y = df_selected["LogSalary"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)


Feature shape: (46224, 4)
Target shape: (46224,)


In [37]:
df_selected["Country"].value_counts().head(10)


Country
United States of America                                11099
Germany                                                  3910
United Kingdom of Great Britain and Northern Ireland     3478
Canada                                                   2054
India                                                    1812
France                                                   1780
Netherlands                                              1350
Poland                                                   1254
Brazil                                                   1222
Australia                                                1188
Name: count, dtype: int64

In [38]:
df_selected["Country"].nunique()


169

# 12. Group Rare Countries

In [39]:
# Get top 10 countries
top_countries = df_selected["Country"].value_counts().head(10).index

# Replace others with "Other"
df_selected["Country"] = df_selected["Country"].apply(
    lambda x: x if x in top_countries else "Other"
)

# Check result
df_selected["Country"].value_counts()


Country
Other                                                   17077
United States of America                                11099
Germany                                                  3910
United Kingdom of Great Britain and Northern Ireland     3478
Canada                                                   2054
India                                                    1812
France                                                   1780
Netherlands                                              1350
Poland                                                   1254
Brazil                                                   1222
Australia                                                1188
Name: count, dtype: int64

# 13. Preprocessing Pipeline


In [42]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Define feature types
numeric_features = ["YearsCodePro"]
categorical_features = ["Country", "EdLevel", "Employment"]

# Create transformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

preprocessor


,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,None
,sparse_output,True


# 14. Train-Test Split

In [43]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)


Training shape: (36979, 4)
Testing shape: (9245, 4)


# 15. Train Baseline Model (Linear Regression)

In [44]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

# Create full pipeline (preprocessing + model)
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

# Train model
model.fit(X_train, y_train)

print("Model training completed.")


Model training completed.


# 16. Model Evaluation (Log Scale)

In [45]:
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# Predict on test set
y_pred_log = model.predict(X_test)

# R² score
r2 = r2_score(y_test, y_pred_log)

# RMSE in log scale
rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log))

print("R² (log scale):", round(r2, 4))
print("RMSE (log scale):", round(rmse_log, 4))


R² (log scale): 0.5273
RMSE (log scale): 0.6427


# 17. Model Evaluation (Original Salary Scale)


In [46]:
# Convert log predictions back to salary
y_test_actual = np.exp(y_test)
y_pred_actual = np.exp(y_pred_log)

# RMSE in USD
rmse_salary = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))

print("RMSE (USD):", round(rmse_salary, 2))


RMSE (USD): 44675.34


# 18. Train Random Forest Model


In [47]:
from sklearn.ensemble import RandomForestRegressor

rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

print("Random Forest training completed.")


Random Forest training completed.


# 19. Random Forest Evaluation (Log Scale)


In [48]:
# Predict
rf_pred_log = rf_model.predict(X_test)

# R²
rf_r2 = r2_score(y_test, rf_pred_log)

# RMSE (log)
rf_rmse_log = np.sqrt(mean_squared_error(y_test, rf_pred_log))

print("Random Forest R² (log):", round(rf_r2, 4))
print("Random Forest RMSE (log):", round(rf_rmse_log, 4))


Random Forest R² (log): 0.521
Random Forest RMSE (log): 0.647


In [49]:
df["YearsCode"].unique()[:10]


array([nan, '18', '27', '12', '6', '21', '4', '5', '20', '14'],
      dtype=object)

In [50]:
df["YearsCode"].unique()


array([nan, '18', '27', '12', '6', '21', '4', '5', '20', '14', '10', '15',
       '11', '3', '24', '8', '13', 'Less than 1 year', '16', '33', '22',
       '30', '32', '7', '35', '28', '40', '17', '29', '19',
       'More than 50 years', '9', '38', '26', '34', '25', '2', '45', '23',
       '31', '43', '1', '48', '41', '50', '39', '42', '37', '36', '44',
       '46', '49', '47'], dtype=object)

In [51]:
df_selected["YearsCode"] = df.loc[df_selected.index, "YearsCode"]

df_selected["YearsCode"].isna().sum()


np.int64(22)

In [52]:
# Remove rows where YearsCode is missing
df_selected = df_selected.dropna(subset=["YearsCode"])

# Clean YearsCode column
def clean_years(value):
    if value == "Less than 1 year":
        return 0.5
    elif value == "More than 50 years":
        return 50
    else:
        return float(value)

df_selected["YearsCode"] = df_selected["YearsCode"].apply(clean_years)

df_selected["YearsCode"].describe()


count    46202.000000
mean        15.639345
std          9.835613
min          0.500000
25%          8.000000
50%         13.000000
75%         20.000000
max         50.000000
Name: YearsCode, dtype: float64

In [53]:
# Redefine feature set including YearsCode
X = df_selected[["YearsCodePro", "YearsCode", "Country", "EdLevel", "Employment"]]
y = df_selected["LogSalary"]

print("New feature shape:", X.shape)


New feature shape: (46202, 5)


In [54]:
numeric_features = ["YearsCodePro", "YearsCode"]
categorical_features = ["Country", "EdLevel", "Employment"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)


In [55]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape, X_test.shape)


(36961, 5) (9241, 5)


In [56]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

model.fit(X_train, y_train)

print("Retrained Linear Regression.")


Retrained Linear Regression.


In [57]:
y_pred_log = model.predict(X_test)

r2 = r2_score(y_test, y_pred_log)
rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log))

print("New R² (log):", round(r2, 4))
print("New RMSE (log):", round(rmse_log, 4))


New R² (log): 0.412
New RMSE (log): 0.7245


In [58]:
X = df_selected[["YearsCodePro", "Country", "EdLevel", "Employment"]]
y = df_selected["LogSalary"]


In [59]:
from sklearn.linear_model import Ridge

ridge_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", Ridge(alpha=1.0))
])

ridge_model.fit(X_train, y_train)

ridge_pred = ridge_model.predict(X_test)

ridge_r2 = r2_score(y_test, ridge_pred)
ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_pred))

print("Ridge R²:", round(ridge_r2, 4))
print("Ridge RMSE:", round(ridge_rmse, 4))


Ridge R²: 0.4119
Ridge RMSE: 0.7245


In [60]:
df["DevType"].isna().sum()


np.int64(12312)

In [61]:
df["DevType"].value_counts().head(10)


DevType
Developer, full-stack                            25735
Developer, back-end                              13745
Developer, front-end                              5071
Developer, desktop or enterprise applications     3904
Other (please specify):                           3080
Developer, mobile                                 2597
Engineering manager                               2033
Student                                           1996
Developer, embedded applications or devices       1845
Data scientist or machine learning specialist     1588
Name: count, dtype: int64

In [62]:
df_selected["DevType"] = df.loc[df_selected.index, "DevType"]
df_selected["DevType"].isna().sum()


np.int64(72)

In [63]:
df_selected = df_selected.dropna(subset=["DevType"])

print("Shape after removing missing DevType:", df_selected.shape)


Shape after removing missing DevType: (46130, 8)


In [64]:
df_selected["DevType"] = df_selected["DevType"].apply(lambda x: x.split(";")[0])

df_selected["DevType"].value_counts().head(10)


DevType
Developer, full-stack                            16541
Developer, back-end                               9199
Developer, front-end                              3146
Developer, desktop or enterprise applications     2371
Developer, mobile                                 1574
Other (please specify):                           1422
Engineering manager                               1250
Developer, embedded applications or devices       1226
DevOps specialist                                  960
Data scientist or machine learning specialist      944
Name: count, dtype: int64

In [65]:
df_selected["DevType"].nunique()


33

In [66]:
# Get top 10 roles
top_roles = df_selected["DevType"].value_counts().head(10).index

# Replace others with "Other"
df_selected["DevType"] = df_selected["DevType"].apply(
    lambda x: x if x in top_roles else "Other"
)

df_selected["DevType"].value_counts()


DevType
Developer, full-stack                            16541
Developer, back-end                               9199
Other                                             7497
Developer, front-end                              3146
Developer, desktop or enterprise applications     2371
Developer, mobile                                 1574
Other (please specify):                           1422
Engineering manager                               1250
Developer, embedded applications or devices       1226
DevOps specialist                                  960
Data scientist or machine learning specialist      944
Name: count, dtype: int64

In [67]:
df_selected["DevType"] = df_selected["DevType"].replace(
    "Other (please specify):", "Other"
)

df_selected["DevType"].value_counts()


DevType
Developer, full-stack                            16541
Developer, back-end                               9199
Other                                             8919
Developer, front-end                              3146
Developer, desktop or enterprise applications     2371
Developer, mobile                                 1574
Engineering manager                               1250
Developer, embedded applications or devices       1226
DevOps specialist                                  960
Data scientist or machine learning specialist      944
Name: count, dtype: int64

In [68]:
df_selected["DevType"].nunique()

10

In [69]:
X = df_selected[[
    "YearsCodePro",
    "Country",
    "EdLevel",
    "Employment",
    "DevType"
]]

y = df_selected["LogSalary"]

print("Feature shape:", X.shape)


Feature shape: (46130, 5)


In [70]:
numeric_features = ["YearsCodePro"]
categorical_features = ["Country", "EdLevel", "Employment", "DevType"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)


In [71]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape, X_test.shape)


(36904, 5) (9226, 5)


In [72]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

model.fit(X_train, y_train)

y_pred_log = model.predict(X_test)

r2 = r2_score(y_test, y_pred_log)
rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log))

print("Updated R² (log):", round(r2, 4))
print("Updated RMSE (log):", round(rmse_log, 4))


Updated R² (log): 0.421
Updated RMSE (log): 0.7121


In [73]:
print(df_selected.shape)
print(X.shape)


(46130, 8)
(46130, 5)


In [74]:
# Baseline again from CURRENT dataset

X_base = df_selected[[
    "YearsCodePro",
    "Country",
    "EdLevel",
    "Employment"
]]

y_base = df_selected["LogSalary"]

X_train, X_test, y_train, y_test = train_test_split(
    X_base, y_base,
    test_size=0.2,
    random_state=42
)

model_base = Pipeline(steps=[
    ("preprocessor", ColumnTransformer(
        transformers=[
            ("num", "passthrough", ["YearsCodePro"]),
            ("cat", OneHotEncoder(handle_unknown="ignore"),
             ["Country", "EdLevel", "Employment"])
        ]
    )),
    ("regressor", LinearRegression())
])

model_base.fit(X_train, y_train)

pred_base = model_base.predict(X_test)

print("Recomputed Baseline R²:",
      round(r2_score(y_test, pred_base), 4))


Recomputed Baseline R²: 0.4138


In [75]:
df["OrgSize"].value_counts().head(10)


OrgSize
20 to 99 employees                                    13380
100 to 499 employees                                  12218
10,000 or more employees                               7929
1,000 to 4,999 employees                               7235
2 to 9 employees                                       6439
10 to 19 employees                                     5254
500 to 999 employees                                   4472
Just me - I am a freelancer, sole proprietor, etc.     4196
5,000 to 9,999 employees                               2677
I don’t know                                           1243
Name: count, dtype: int64

In [76]:
df_selected["OrgSize"] = df.loc[df_selected.index, "OrgSize"]
df_selected["OrgSize"].isna().sum()


np.int64(21)

In [77]:
df_selected = df_selected.dropna(subset=["OrgSize"])

print("Shape after removing missing OrgSize:", df_selected.shape)


Shape after removing missing OrgSize: (46109, 9)


In [78]:
df_selected["OrgSize"].nunique()


10

In [79]:
X = df_selected[[
    "YearsCodePro",
    "Country",
    "EdLevel",
    "Employment",
    "DevType",
    "OrgSize"
]]

y = df_selected["LogSalary"]

print("Feature shape:", X.shape)


Feature shape: (46109, 6)


In [80]:
numeric_features = ["YearsCodePro"]
categorical_features = [
    "Country",
    "EdLevel",
    "Employment",
    "DevType",
    "OrgSize"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)


In [81]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)


In [82]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

model.fit(X_train, y_train)

y_pred_log = model.predict(X_test)

r2 = r2_score(y_test, y_pred_log)
rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log))

print("R² with OrgSize:", round(r2, 4))
print("RMSE with OrgSize:", round(rmse_log, 4))


R² with OrgSize: 0.4369
RMSE with OrgSize: 0.7001


In [83]:
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

rf_r2 = r2_score(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))

print("Random Forest R²:", round(rf_r2, 4))
print("Random Forest RMSE:", round(rf_rmse, 4))


Random Forest R²: 0.4085
Random Forest RMSE: 0.7175


In [84]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    model,
    X,
    y,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

print("Cross-validated R² scores:", cv_scores)
print("Mean CV R²:", round(cv_scores.mean(), 4))


Cross-validated R² scores: [0.39357792 0.42300887 0.45912761 0.43585435 0.46649546]
Mean CV R²: 0.4356


In [85]:
final_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

final_model.fit(X, y)

print("Final model trained on full dataset.")


Final model trained on full dataset.
